<a href="https://www.kaggle.com/code/adegbaju/acoustic-species-identification-pipeline?scriptVersionId=303484666" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# BirdCLEF+ 2026 – Acoustic Species Identification Pipeline

# Setup loading

In [1]:
!pip install torch tqdm

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import torchaudio.functional as F
import timm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Configuration 

In [3]:
CFG = {
    'sample_rate': 32000,
    'segment_duration': 5,
    'n_mels': 128,
    'hop_length': 512,
    'n_fft': 2048,
    'f_min': 0,
    'f_max': 16000,
    'batch_size': 32,
    'epochs': 20,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'num_workers': 4,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'model_name': 'efficientnet_b0',
    'seed': 42,
    'train_audio_dir': '/kaggle/input/competitions/birdclef-2026/train_audio',
    'train_soundscapes_dir': '/kaggle/input/competitions/birdclef-2026/train_soundscapes',
    'test_dir': '/kaggle/input/competitions/birdclef-2026/test_soundscapes',
    'taxonomy_path': '/kaggle/input/competitions/birdclef-2026/taxonomy.csv',
    'train_csv_path': '/kaggle/input/competitions/birdclef-2026/train.csv',
    'soundscape_labels_path': '/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv',
    'model_save_path': 'birdclef2026_model.pth',
    'submission_path': 'submission.csv'
}

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])

taxonomy = pd.read_csv(CFG['taxonomy_path'])
species_list = taxonomy['primary_label'].tolist()
species_to_idx = {sp: i for i, sp in enumerate(species_list)}
num_classes = len(species_list)

# Prepare training samples

In [4]:
# 1. From train_audio (individual clips)
train_df = pd.read_csv(CFG['train_csv_path'])
audio_samples = []
for idx, row in train_df.iterrows():
    filename = row['filename']
    primary = row['primary_label']
    secondary = row['secondary_labels']
    # secondary_labels may be NaN or empty string; if present, split by ';'
    labels = [primary]
    if pd.notna(secondary) and secondary.strip():
        labels.extend(secondary.split(';'))
    # Convert to indices, ignoring any that are not in taxonomy (shouldn't happen)
    label_indices = [species_to_idx[lab] for lab in labels if lab in species_to_idx]
    audio_samples.append({
        'file_path': os.path.join(CFG['train_audio_dir'], filename),
        'start': None,          # full clip, no crop start
        'end': None,
        'labels': label_indices
    })


In [5]:
def time_to_seconds(t):
    if isinstance(t, (int, float)):
        return float(t)
    parts = str(t).split(':')
    if len(parts) == 3:
        h, m, s = parts
        return int(h)*3600 + int(m)*60 + float(s)
    return float(t)

In [6]:
# 2. From train_soundscapes (annotated 5-second segments)
soundscape_labels = pd.read_csv(CFG['soundscape_labels_path'])
soundscape_samples = []

for idx, row in soundscape_labels.iterrows():
    filename = row['filename']
    
    start = time_to_seconds(row['start'])
    end = time_to_seconds(row['end'])

    labels = row['primary_label'].split(';')
    label_indices = [species_to_idx[lab] for lab in labels if lab in species_to_idx]

    soundscape_samples.append({
        'file_path': os.path.join(CFG['train_soundscapes_dir'], filename),
        'start': start,
        'end': end,
        'labels': label_indices
    })

In [7]:
# Combine all samples
all_samples = audio_samples + soundscape_samples
print(f"Total training samples: {len(all_samples)}")

# Split into train / validation (stratified by primary label not feasible, so random)
train_samples, val_samples = train_test_split(all_samples, test_size=0.2, random_state=CFG['seed'])
print(f"Train samples: {len(train_samples)}, Val samples: {len(val_samples)}")

Total training samples: 37027
Train samples: 29621, Val samples: 7406


# Dataset Class

In [8]:
class BirdCLEFDataset(Dataset):
    def __init__(self, samples, is_train=True):
        self.samples = samples
        self.is_train = is_train
        self.sr = CFG['sample_rate']
        self.seg_len = CFG['segment_duration'] * self.sr   # number of samples
        # Mel-spectrogram transform (will be applied to waveform)
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.sr,
            n_fft=CFG['n_fft'],
            hop_length=CFG['hop_length'],
            n_mels=CFG['n_mels'],
            f_min=CFG['f_min'],
            f_max=CFG['f_max']
        )
        # Amplitude to dB
        self.to_db = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        file_path = sample['file_path']
        start, end = sample['start'], sample['end']
        labels = sample['labels']

        # Load audio
        waveform, sr = torchaudio.load(file_path)
        if sr != self.sr:
            waveform = F.resample(waveform, sr, self.sr)
        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Extract the desired segment
        if start is not None and end is not None:
            # Annotated segment from soundscape
            start_sample = int(start * self.sr)
            end_sample = start_sample + self.seg_len
            waveform = waveform[:, start_sample:end_sample]
        else:
            # train_audio clip: we want exactly 5 seconds
            # If longer, take a random 5-second crop during training, else pad
            pass

        # Now waveform may be shorter or longer than 5 seconds
        current_len = waveform.shape[1]
        target_len = self.seg_len

        if current_len > target_len:
            # Random crop during training, center crop during validation
            if self.is_train:
                start_idx = np.random.randint(0, current_len - target_len + 1)
            else:
                start_idx = (current_len - target_len) // 2
            waveform = waveform[:, start_idx:start_idx + target_len]
        elif current_len < target_len:
            # Pad with zeros
            pad = target_len - current_len
            waveform = torch.nn.functional.pad(waveform, (0, pad))

        # Now waveform shape: (1, target_len)

        # Compute mel-spectrogram
        mel = self.mel_spec(waveform)          # (1, n_mels, time)
        mel = self.to_db(mel)                  # convert to dB scale
        # Normalize to [0,1] roughly (min/max over dataset could be used, but simple scaling)
        mel = (mel + 80) / 80                   # typical: dB range -80 to 0 → 0 to 1
        mel = mel.clamp(0, 1)

        # Duplicate to 3 channels (for ImageNet pretrained models)
        mel_3ch = mel.repeat(3, 1, 1)           # (3, n_mels, time)

        # Create label vector (multi-hot)
        label_vec = torch.zeros(num_classes, dtype=torch.float32)
        for lab in labels:
            label_vec[lab] = 1.0

        return mel_3ch, label_vec

#  DataLoaders

In [9]:
train_dataset = BirdCLEFDataset(train_samples, is_train=True)
val_dataset = BirdCLEFDataset(val_samples, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'],
                          shuffle=True, num_workers=CFG['num_workers'])
val_loader = DataLoader(val_dataset, batch_size=CFG['batch_size'],
                        shuffle=False, num_workers=CFG['num_workers'])

# model

In [10]:
class BirdCLEFModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Use EfficientNet-B0 pretrained on ImageNet
        self.backbone = timm.create_model(CFG['model_name'], pretrained=True, num_classes=0)
        # Global average pooling (already included in timm models)
        self.fc = nn.Linear(self.backbone.num_features, num_classes)

    def forward(self, x):
        # x: (batch, 3, H, W)
        features = self.backbone(x)   # (batch, num_features)
        logits = self.fc(features)    # (batch, num_classes)
        return logits

model = BirdCLEFModel(num_classes).to(CFG['device'])

# Loss: weighted BCE to handle class imbalance (optional)
# Compute class weights based on frequency in training set
pos_counts = np.zeros(num_classes)
for sample in train_samples:
    for lab in sample['labels']:
        pos_counts[lab] += 1
# Avoid zero
pos_counts = np.maximum(pos_counts, 1)
neg_counts = len(train_samples) - pos_counts
# Weight = negative/positive (to give more weight to rare classes)
class_weights = torch.tensor(neg_counts / pos_counts, dtype=torch.float32).to(CFG['device'])

criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'])


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

# Training Loop

In [11]:
best_val_loss = float('inf')
for epoch in range(1, CFG['epochs'] + 1):
    model.train()
    train_loss = 0.0
    for inputs, labels in tqdm(train_loader, desc=f'Epoch {epoch} Train'):
        inputs, labels = inputs.to(CFG['device']), labels.to(CFG['device'])
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
    train_loss /= len(train_loader.dataset)

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f'Epoch {epoch} Val'):
            inputs, labels = inputs.to(CFG['device']), labels.to(CFG['device'])
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
    val_loss /= len(val_loader.dataset)

    scheduler.step()

    print(f'Epoch {epoch}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CFG['model_save_path'])
        print(f'Model saved with val loss {val_loss:.4f}')

print('Training completed.')


Epoch 1 Val: 100%|██████████| 232/232 [01:57<00:00,  1.97it/s]


Epoch 1: Train Loss: 1.2855, Val Loss: 1.1583
Model saved with val loss 1.1583


Epoch 2 Val: 100%|██████████| 232/232 [01:48<00:00,  2.14it/s]


Epoch 2: Train Loss: 1.0365, Val Loss: 1.0867
Model saved with val loss 1.0867


Epoch 3 Val: 100%|██████████| 232/232 [01:58<00:00,  1.97it/s]


...

Epoch 18: Train Loss: 0.3078, Val Loss: 1.1160


Epoch 19 Val: 100%|██████████| 232/232 [01:42<00:00,  2.26it/s]


Epoch 19: Train Loss: 0.2984, Val Loss: 1.1126


Epoch 20 Val: 100%|██████████| 232/232 [01:43<00:00,  2.25it/s]

Epoch 20: Train Loss: 0.2970, Val Loss: 1.1236
Training completed.


# Inference (for submission)

In [12]:
def predict_test(model, test_dir, device):
    model.eval()
    test_files = [f for f in os.listdir(test_dir) if f.endswith('.ogg')]
    results = []

    # Mel-spectrogram transform (same as training)
    mel_spec = torchaudio.transforms.MelSpectrogram(
        sample_rate=CFG['sample_rate'],
        n_fft=CFG['n_fft'],
        hop_length=CFG['hop_length'],
        n_mels=CFG['n_mels'],
        f_min=CFG['f_min'],
        f_max=CFG['f_max']
    ).to(device)
    to_db = torchaudio.transforms.AmplitudeToDB().to(device)

    for fname in tqdm(test_files, desc='Predicting test soundscapes'):
        file_path = os.path.join(test_dir, fname)
        waveform, sr = torchaudio.load(file_path)
        if sr != CFG['sample_rate']:
            waveform = F.resample(waveform, sr, CFG['sample_rate'])
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Split into 5-second segments (non-overlapping)
        total_samples = waveform.shape[1]
        seg_samples = CFG['segment_duration'] * CFG['sample_rate']
        num_segments = total_samples // seg_samples   # should be 12

        for seg_idx in range(num_segments):
            start = seg_idx * seg_samples
            end = start + seg_samples
            segment = waveform[:, start:end]   # (1, seg_samples)

            # Compute mel-spectrogram
            mel = mel_spec(segment)            # (1, n_mels, time)
            mel = to_db(mel)
            mel = (mel + 80) / 80
            mel = mel.clamp(0, 1)
            mel_3ch = mel.repeat(3, 1, 1)      # (3, n_mels, time)

            # Add batch dimension
            input_tensor = mel_3ch.unsqueeze(0).to(device)   # (1, 3, H, W)

            with torch.no_grad():
                logits = model(input_tensor)                 # (1, num_classes)
                probs = torch.sigmoid(logits).cpu().numpy().flatten()

            # row_id: filename without extension + end time
            end_time = (seg_idx + 1) * CFG['segment_duration']
            row_id = f"{os.path.splitext(fname)[0]}_{end_time}"
            results.append([row_id] + probs.tolist())

    # Create submission dataframe
    columns = ['row_id'] + species_list
    submission = pd.DataFrame(results, columns=columns)
    return submission

# Load best model
model.load_state_dict(torch.load(CFG['model_save_path']))
model.to(CFG['device'])

# Run inference (test_dir will be populated at submission)
if os.path.exists(CFG['test_dir']):
    submission = predict_test(model, CFG['test_dir'], CFG['device'])
    submission.to_csv(CFG['submission_path'], index=False)
    print(f'Submission saved to {CFG['submission_path']}')
else:
    print('Test directory not found. Skipping inference.')

Predicting test soundscapes: 0it [00:00, ?it/s]

Submission saved to submission.csv
